# Attribute extraction pipeline
## Data-driven extraction

In [ ]:
%env GENAI_API_KEY=...
%env OPENAI_API_KEY=...
%env UFAL_API_KEY=...
%env GROQ_API_KEY=...

In [693]:
def get_llm(provider: str, model: str = None) -> LLMClient:
    if provider == "openai":
        return OpenAIClient(model=model) if model else OpenAIClient()

    elif provider == "genai":
        return GenAIClient(model=model) if model else GenAIClient()

    elif provider == "ufalai":
        return UfalClient(model=model) if model else UfalClient()

    elif provider == "groq":
        return GroqClient(model=model) if model else GroqClient()

    else:
        raise ValueError(f"Unknown provider: {provider}")

In [2]:
import json 
def get_json(string):
    stripped = ""
    braces = 0
    for s in string:
        if s == "{":
            braces += 1
        if braces >= 1:
            stripped += s
        if s == "}":
            braces -= 1
    return json.loads(stripped, strict=False)


In [3]:
def retry(func, *args, retries=1, delay=30, **kwargs):
    """
    Call func with retries and delay on Exception.
    """
    for attempt in range(retries + 1):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if attempt == retries:
                # Last attempt failed, re-raise
                raise
            print(f"Error: {e}. {func} Another attempt...")
            time.sleep(delay)

## EITHER Load dev data

In [94]:
dataset_name = "../data/categorization/DKK-dev-final.csv" 

In [95]:
import pandas as pd
import csv
import sys
csv.field_size_limit(sys.maxsize)

df = pd.read_csv(
    dataset_name,
    sep=",",
    quoting=1,  # csv.QUOTE_ALL
    quotechar='"',
    escapechar="\\",
    engine="python"  # Required when dealing with multiline quoted fields
)

# Check the first few rows
marenky = df[df["judgement_model_short"] == "A"]
pod_vlivem = df[df["judgement_model_short"] == "B"]
s_nasledkem = df[df["judgement_model_short"] == "C"]
jina = df[df["judgement_model_short"] == "D"]
nedopravni = df[df["judgement_model_short"] == "E"]

In [96]:
print(f"Mařenka: {len(marenky)}")
print(f"Řízení pod vlivem omamných a psychotropních látek bez následku: {len(pod_vlivem)}")
print(f"Dopravní kriminalita s následkem : {len(s_nasledkem)}")
print(f"Dopravní kriminalita jiná: {len(jina)}")
print(f"Není dopravní kriminalita: {len(nedopravni)}")


Mařenka: 38
Řízení pod vlivem omamných a psychotropních látek bez následku: 156
Dopravní kriminalita s následkem : 92
Dopravní kriminalita jiná: 1
Není dopravní kriminalita: 895


## OR Load eval data

In [97]:
df = pd.read_csv("../data/evaluation/annotation_with_complete_with_curation_and_llm.csv", encoding="utf-8-sig")
df = df.sort_values(by="id").reset_index(drop=True)
df = df[df["id"] != 115839385]
df = df.rename(columns={"id": "judgement_slt_id"})
df = df.rename(columns={"text": "judgement_factual_sentence"})
pod_vlivem=df

## Extraction

In [252]:
attribute_schema = {
    "substanceUsed": {
        "description": "Drug or substance detected",
        "cardinality": "multiple"
    },
    "amountUsed": {
        "description": "The amount of drug or substance detected",
        "cardinality": "multiple"
    },
    "measuringMethod": {
        "description": "Type of measurement taken",
        "cardinality": "multiple"
    },
    "measuringTime": {
        "description": "Time of measurement",
        "cardinality": "multiple"
    },
    "measuringDevice": {
        "description": "Measuring device used for testing",
        "cardinality": "single"
    },
    "refusedTesting": {
        "description": "Information that the driver refused testing",
        "cardinality": "single"
    },
    "dateOfOffense": {
        "description": "Date when the offense occurred",
        "cardinality": "single"
    },
    "timeOfOffense": {
        "description": "Time when the offense occurred",
        "cardinality": "single"
    },
    "placeOfOffense": {
        "description": "Name of the city/municipality where the offense occurred",
        "cardinality": "single"
    },
    "geographicalDetails": {
        "description": "City, municipality, direction, or geographic place used to show the offense on a map",
        "cardinality": "single"
    },
    "placeDetails": {
        "description": "More specific description of the place, e.g. parking lot or intersection",
        "cardinality": "single"
    },
    "vehicleModel": {
        "description": "Vehicle brand and model involved in the offense, e.g. Skoda Octavia, or 'anonymized' if only capital letters are provided",
        "cardinality": "single"
    },
    "vehicleType": {
        "description": "Type of vehicle involved in the offense",
        "cardinality": "single"
    },
    "checkedBy": {
        "description": "Who conducted the inspection or check",
        "cardinality": "single"
    },
    "typeOfRoad": {
        "description": "Type of road where the offense occurred",
        "cardinality": "single"
    },
    "reasonForCheck": {
        "description": "Reason why the inspection decided to stop the car",
        "cardinality": "single"
    },
    "licenseStatus": {
        "description": "Validity status of the driver's license",
        "cardinality": "single"
    },
    "lawBroken": {
        "description": "Reference to a law that was broken by the offense",
        "cardinality": "single"
    }
}


In [264]:
classification_schema = {

  "substanceUsed": {
    "alcohol": "Alcohol or ethanol consumption",
    "cannabis": "Cannabis or marijuana",
    "methamphetamine": "Methamphetamine (pervitin)",
    "amphetamine": "Amphetamine",
    "cocaine": "Cocaine",
    "mdma": "MDMA or ecstasy",
    "hashish": "Hashish (cannabis resin)",
    "morphine": "Morphine",
    "barbiturate": "Barbiturates",
    "benzodiazepine": "Benzodiazepines",
    "methadone": "Methadone",
    "heroin": "Heroin",
    "other": "Any other substance not listed above"
  },

  "measuringMethod": {
    "breath": "Measurement using a breathalyzer",
    "blood": "Measurement using a blood sample",
    "urine": "Measurement using a urine sample",
    "other": "Other measurement method"
  },

  "vehicleType": {
    "car": "Passenger car",
    "van": "Van or light commercial vehicle",
    "truck": "Truck or heavy goods vehicle",
    "bus": "Bus",
    "motorcycle": "Motorcycle",
    "other": "Other type of vehicle"
  },

  "typeOfRoad": {
    "urban": "Road within a municipality or urban area",
    "motorway": "Motorway / highway",
    "class_I": "First class road",
    "class_II": "Second class road",
    "class_III": "Third class road",
    "other": "Other type of road"
  },

  "reasonForCheck": {
    "traffic_accident": "Check initiated due to a traffic accident",
    "speeding": "Vehicle stopped due to speeding",
    "traffic_violation": "Other traffic violation",
    "suspected_intoxication": "Police suspected intoxication",
    "traffic_check": "Routine traffic control",
    "other": "Other reason for the check"
  },

  "licenseStatus": {
    "valid": "Driver had a valid license",
    "banned": "Driving despite a ban or revoked license",
    "none": "License never issued"
  },
  "refusedTesting": {
    "TRUE": "Driver refused to take the drug exam",
    "FALSE": "Driving despite a ban or revoked license",
  }

}

formatting_rules = {
    "dateOfOffense": "DD.MM.YYYY",
    "timeOfOffense": "HH:MM",
    "measuringTime": "HH:MM",
    "amountUsed": "[float] [unit]",
    "placeOfOffense": "name of city/municipality in nominative"
  }


In [254]:
def format_attribute_schema(attribute_list):
    lines = [f"{key} ({value['cardinality']}): {value['description']}" for key, value in attribute_list.items()]
    return "\n".join(lines)
        

In [255]:
def format_value_schema(schema):
    sections = []
    for field, classes in schema.items():
        lines = [f"{field}:"]
        for label, description in classes.items():
            lines.append(f"- {label}: {description}")
        sections.append("\n".join(lines))

    return "\n\n".join(sections)

In [262]:
def format_formatting_rules(formatting_rules):
    lines = [f"{key} [{value}]" for key, value in formatting_rules.items()]
    return "\n".join(lines)

In [446]:
def tag_sentence(id, sentence, attribute_schema):
    system_prompt = f"""
You are an information extraction agent.

Your task is to insert XML-like tags into the input text.

Use only the tag names defined below. Do not use any other tags. Return only the tagged text. Do not explain anything.

GENERAL RULES

Preserve the original text exactly.
Insert tags directly around the relevant span.
Do not rewrite, normalize, or translate the text.
Tags must not overlap, cross, or nest.
Tag only the minimal span that expresses the information.
Do not include redundant words such as “on”, “around”, “at the time”, etc., unless they are part of the meaning.
If several spans seem possible, choose the shortest span that still fully expresses the target attribute.
A single tagged span must represent one attribute only.
Tag only information explicitly stated in the text. Do not infer missing information.
If no listed attribute is present for a span, do not tag it.

SINGLE VS MULTIPLE TAGS

If an attribute is marked as "single", use that tag at most once in the text. If the information appears multiple times, tag only the first occurrence. If the information is expressed across a longer phrase, tag only the first continuous relevant span.

If an attribute is marked as "multiple", tag each distinct occurrence separately. If the same information is repeated, do not tag it again.

ANONYMIZED VALUES

If a value is anonymized, it must still be tagged. Examples include masked dates such as XX.XX.XXXX, initials such as O. T., or placeholders such as anonymized. Include punctuation as part of the tagged span when it is part of the anonymized value.

ATTRIBUTES TO TAG

{format_attribute_schema(attribute_schema)}

EXAMPLES

on <dateOfOffense>04.01.2022</dateOfOffense> he was driving a motor vehicle
Do not include extra words such as “on”.

he was driving a passenger vehicle <vehicleModel>Škoda Octavia</vehicleModel>
Tag only the vehicle model.

after previously consuming a large amount of <substanceUsed>alcohol</substanceUsed>
Tag only the substance.

on a <typeOfRoad>I. class</typeOfRoad> road in the direction of Prešov
Include the road class designation.

a driver who had a <licenseStatus>ban on driving motor vehicles</licenseStatus> imposed
Tag the phrase that clearly indicates the prohibition.

on <dateOfOffense>XX.XX.XXXX</dateOfOffense> he was driving a motor vehicle
Even anonymized values must be tagged.

EXAMPLE TAGGED TEXT

dňa <dateOfOffense>25. januára 2018</dateOfOffense> v čase okolo <timeOfOffense>02.50 h</timeOfOffense> v Stupave na Železničnej ulici, ako vodič viedol <vehicleType>osobné motorové vozidlo</vehicleType> značky <vehicleModel>O. T.</vehicleModel>, pričom u vodiča bola vykonaná <measuringMethod>dychová skúška</measuringMethod> prístrojom <measuringDevice>AlcoQuant 6020 plus</measuringDevice>, ktorou bola zistená hodnota <amountUsed>0,87 mg/l</amountUsed> o <measuringTime>02.55 h</measuringTime> a následne opakovaným meraním hodnota <amountUsed>0,82 mg/l</amountUsed> o <measuringTime>03.10 h</measuringTime>.


"""
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": sentence},
    ]
    tagged = LLM.chat(messages)


    return tagged


In [815]:
def tag_sentence_complex(id, sentence, attribute_schema, classification_schema, formatting_rules):
    system_prompt = f"""
You are an information extraction and normalization agent.

Your task is to insert XML-like tags into the input text and, when applicable, add tag properties class and/or value to the opening tag.

Use only the tag names defined below. Do not use any other tags. Return only the tagged text. Do not explain anything.

GENERAL RULES

Preserve the original text exactly.
Insert tags directly around the relevant span.
Do not rewrite, translate, or otherwise change the text content.
Tags must not overlap, cross, or nest.
Tag only the minimal span that expresses the information.
Do not include redundant words such as "on", "around", "at the time", etc., unless they are part of the meaning.
If several spans seem possible, choose the shortest span that still fully expresses the target attribute.
A single tagged span must represent one attribute only.
Tag only information explicitly stated in the text. Do not infer missing information.
If no listed attribute is present for a span, do not tag it.
Only add tag properties to opening tags.
Do not add both class and value unless explicitly required. Normally a tag gets only one of them.

SINGLE VS MULTIPLE TAGS

If an attribute is marked as "single":
Use that tag at most once in the text.
If the information appears multiple times, tag only the first occurrence.
If the information is expressed across a longer phrase, tag only the first continuous relevant span.
If an attribute is marked as "multiple":

Tag each distinct occurrence separately.

If the same information is repeated, do not tag it again.

ANONYMIZED VALUES

If a value is anonymized, it must still be tagged.

Examples include masked dates such as XX.XX.XXXX, initials such as O. T., or placeholders such as anonymized.
Include punctuation as part of the tagged span when it is part of the anonymized value.
If the tagged content is anonymized and no class rule is defined for that tag, add value="anonymized".

DECISION ORDER FOR TAG PROPERTIES

For each tagged entity:
If the tag name has a classification schema, add class="..."
Else if the tag name has a formatting rule, add value="..."
Else if the tagged content is anonymized and the tag has no classification rule, add value="anonymized"
Else add no tag property

ATTRIBUTES TO TAG

{format_attribute_schema(attribute_schema)}

CLASSIFICATION SCHEMA

{format_value_schema(classification_schema)}

CLASSIFICATION INSTRUCTIONS

Choose exactly one class from the allowed categories for that tag.
Use tag content and local context if needed.
If no category matches clearly, use class="other" only when that category exists.
Otherwise add no class property.

FORMATTING RULES

{format_formatting_rules(formatting_rules)}

FORMATTING INSTRUCTIONS

Keep the original tagged text unchanged.
Add value only as a tag property in the opening tag.

EXAMPLES

on <dateOfOffense value="04.01.2022">04.01.2022</dateOfOffense> he was driving a motor vehicle
Do not include extra words such as "on".

he was driving a passenger vehicle <vehicleModel>Škoda Octavia</vehicleModel>
Tag only the vehicle model.

after previously consuming a large amount of <substanceUsed class="alcohol">alcohol</substanceUsed>
Tag only the substance.

on a <typeOfRoad class="class_I">I. class</typeOfRoad> road in the direction of Prešov
Include the road class designation.

a driver who had a <licenseStatus class="banned">ban on driving motor vehicles</licenseStatus> imposed
Tag the phrase that clearly indicates the prohibition.

on <dateOfOffense value="anonymized">XX.XX.XXXX</dateOfOffense> he was driving a motor vehicle
Even anonymized values must be tagged.

EXAMPLE TAGGED TEXT

dňa <dateOfOffense value="25.01.2018">25. januára 2018</dateOfOffense> v čase okolo <timeOfOffense value="02:50">02.50 h</timeOfOffense> v Stupave na Železničnej ulici, ako vodič viedol <vehicleType class="car">osobné motorové vozidlo</vehicleType> značky <vehicleModel value="anonymized">O. T.</vehicleModel>, pričom u vodiča bola vykonaná <measuringMethod class="breath">dychová skúška</measuringMethod> prístrojom <measuringDevice>AlcoQuant 6020 plus</measuringDevice>, ktorou bola zistená hodnota <amountUsed value="0.87 mg/l">0,87 mg/l</amountUsed> o <measuringTime value="02:55">02.55 h</measuringTime> a následne opakovaným meraním hodnota <amountUsed value="0.82 mg/l">0,82 mg/l</amountUsed> o <measuringTime value="03:10">03.10 h</measuringTime>.
"""

    print(system_prompt)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": sentence},
    ]
    tagged = LLM.chat(messages)


    return tagged


In [448]:
def tag_sentence_complex_with_structure(id, sentence, attribute_schema,  classification_schema, formatting_rules):
    system_prompt = f"""
You are an information extraction and normalization agent.

Your task is to insert XML-like tags into the input text to mark specific attributes and events.

Return only the tagged text.
Do not explain anything.

GENERAL RULES

Preserve the original text exactly.
Do not rewrite, translate, or modify any text.
Insert tags directly around the relevant span.
Tags must not overlap or cross.
Nesting is allowed only when grouping related attributes inside a container tag.
Tag the shortest span that expresses the information.
Tag only information explicitly stated in the text. Do not infer missing data.
If an attribute appears multiple times with the same value within the same event, tag it only once.
Only add properties to opening tags.

TAG PROPERTIES

Some tags require class="..." when a classification schema exists.
Some tags require value="..." when a formatting rule exists.
If the tagged content is anonymized and no class rule exists, add value="anonymized".

CONTAINER TAGS (FOR GROUPING)

priorDecision
offense
measurement
vehicle
location
sanction
relatedOffense

Container tags group related attributes belonging to one event or object.

Container tags may contain other tags.
Attribute tags must not contain other tags.
Each offense described in the text should have its own offense container.
Each measurement result should have its own measurement container.

Preferred structure when possible:

priorDecision may contain sanction and relatedOffense
relatedOffense may contain dateOfOffense, substanceUsed, and measurement
offense may contain dateOfOffense, timeOfOffense, location, vehicle, measurement, reasonForCheck, checkedBy, refusedTesting, lawBroken, licenseStatus
location may contain placeOfOffense, geographicalDetails, placeDetails, typeOfRoad
vehicle may contain vehicleType and vehicleModel
measurement may contain measuringTime, amountUsed, substanceUsed, measuringMethod, measuringDevice

If grouping is unclear, tag only the attributes.

ATTRIBUTE TAGS

{format_attribute_schema(attribute_schema)}

FORMATTING RULES

{format_formatting_rules(formatting_rules)}

The tagged text itself must remain unchanged.

CLASSIFICATION SCHEMAS

{format_value_schema(classification_schema)}

Choose exactly one class when a schema exists.
If no category matches clearly, use class="other" only if that category exists.

STRUCTURE PLANNING (DO NOT OUTPUT)

Before writing the final tagged text:
Identify offense events described in the text.
Identify measurement events and assign them to the correct offense.
Identify vehicle, location, and sanction information and assign them to the correct event.
Determine the container structure using the allowed container tags.
Verify that tags will not overlap or cross.
Do not output this reasoning.

FINAL VALIDATION

Before returning the result verify that:

every opening tag has a closing tag
tags do not overlap or cross
container tags contain only valid attributes
the original text has not been modified
If an error is detected, correct it before producing the output.
When multiple measurements are present, always create a separate measurement container for each measurement result.
Return only the tagged text.

EXAMPLE
When an example is provided, follow the same tagging structure and container hierarchy.

Input text:

dňa 25. januára 2018 v čase okolo 02.50 h v Stupave vodič viedol osobné motorové vozidlo značky O. T., pričom bola vykonaná dychová skúška prístrojom AlcoQuant 6020 plus s výsledkom 0,87 mg/l o 02.55 h a následne 0,82 mg/l o 03.10 h.

Output:

<offense>dňa <dateOfOffense value="25.01.2018">25. januára 2018</dateOfOffense> v čase okolo <timeOfOffense value="02:50">02.50 h</timeOfOffense> v <location><placeOfOffense>Stupave</placeOfOffense></location> vodič viedol <vehicle><vehicleType class="car">osobné motorové vozidlo</vehicleType> značky <vehicleModel value="anonymized">O. T.</vehicleModel></vehicle>, pričom bola vykonaná <measurement><measuringMethod class="breath">dychová skúška</measuringMethod> prístrojom <measuringDevice>AlcoQuant 6020 plus</measuringDevice> s výsledkom <amountUsed value="0.87 mg/l">0,87 mg/l</amountUsed> o <measuringTime value="02:55">02.55 h</measuringTime></measurement> a následne <measurement><amountUsed value="0.82 mg/l">0,82 mg/l</amountUsed> o <measuringTime value="03:10">03.10 h</measuringTime></measurement>.</offense>
"""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": sentence},
    ]
    tagged = LLM.chat(messages)


    return tagged


In [449]:
def add_values_and_structure_to_tags(id, sentence,  classification_schema, formatting_rules):
    system_prompt = f"""
You are an information extraction and normalization agent.

ou will receive text containing XML-like entity tags, for example:
<vehicleType>...</vehicleType>

Your task is to enrich existing opening tags by adding tag properties based on the rules below and add container tags for grouping.

GENERAL RULES
- Preserve the original text exactly.
- Do not rewrite text outside tags.
- Do not change tag names.
- Do not remove tags.
- Do not change closing tags.
- Only add tag properties to opening tags.
- Add container tags without properties.
- Return only the transformed text.
- Do not explain anything.

CONTAINER TAGS (FOR GROUPING)

priorDecision
offense
measurement
vehicle
location
sanction
relatedOffense

Container tags group related attributes belonging to one event or object.

Container tags may contain other tags.
Attribute tags must not contain other tags.
Each offense described in the text should have its own offense container.
Each measurement result should have its own measurement container.

Preferred structure when possible:

priorDecision may contain sanction and relatedOffense
relatedOffense may contain dateOfOffense, substanceUsed, and measurement
offense may contain dateOfOffense, timeOfOffense, location, vehicle, measurement, reasonForCheck, checkedBy, refusedTesting, lawBroken, licenseStatus
location may contain placeOfOffense, geographicalDetails, placeDetails, typeOfRoad
vehicle may contain vehicleType and vehicleModel
measurement may contain measuringTime, amountUsed, substanceUsed, measuringMethod, measuringDevice

DECISION ORDER FOR ADDING PROPERTIES
For each tagged entity:
1. If the tag name is in the classification schema, add class="..."
2. Else if the tag name is in the formatting rules, add value="..."
3. Else if the content is anonymized and the tag has no classification rule, add value="anonymized"
4. Else add nothing

Do not add both class and value unless explicitly required. Normally a tag gets only one of them.

CLASSIFICATION SCHEMA

{format_value_schema(classification_schema)}

CLASSIFICATION INSTRUCTIONS
- Choose exactly one class from the allowed categories for that tag.
- Use tag content and local context if needed.
- If no category matches clearly, use class="other" only when that category exists.
- Otherwise add no class property.

FORMATTING RULES

{format_formatting_rules(formatting_rules)}

FORMATTING INSTRUCTIONS
- Add tag property value with the normalized value.
- Keep the original tagged text unchanged.

ANONYMIZATION RULE
- If the content is anonymized and the tag has no classification rule, add value="anonymized"
- Examples include initials, masked names, redacted strings, placeholders, or obvious anonymization markers

STRUCTURE PLANNING (DO NOT OUTPUT)

Before writing the final tagged text:
Identify offense events described in the text.
Identify measurement events and assign them to the correct offense.
Identify vehicle, location, and sanction information and assign them to the correct event.
Determine the container structure using the allowed container tags.
Verify that tags will not overlap or cross.
Do not output this reasoning.

FINAL VALIDATION

Before returning the result verify that:

every opening tag has a closing tag
tags do not overlap or cross
container tags contain only valid attributes
the original text has not been modified
If an error is detected, correct it before producing the output.
When multiple measurements are present, always create a separate measurement container for each measurement result.
Return only the tagged text.

EXAMPLE
When an example is provided, follow the same tagging structure and container hierarchy.

Input text:

dňa 25. januára 2018 v čase okolo 02.50 h v Stupave vodič viedol osobné motorové vozidlo značky O. T., pričom bola vykonaná dychová skúška prístrojom AlcoQuant 6020 plus s výsledkom 0,87 mg/l o 02.55 h a následne 0,82 mg/l o 03.10 h.

Output:

<offense>dňa <dateOfOffense value="25.01.2018">25. januára 2018</dateOfOffense> v čase okolo <timeOfOffense value="02:50">02.50 h</timeOfOffense> v <location><placeOfOffense>Stupave</placeOfOffense></location> vodič viedol <vehicle><vehicleType class="car">osobné motorové vozidlo</vehicleType> značky <vehicleModel value="anonymized">O. T.</vehicleModel></vehicle>, pričom bola vykonaná <measurement><measuringMethod class="breath">dychová skúška</measuringMethod> prístrojom <measuringDevice>AlcoQuant 6020 plus</measuringDevice> s výsledkom <amountUsed value="0.87 mg/l">0,87 mg/l</amountUsed> o <measuringTime value="02:55">02.55 h</measuringTime></measurement> a následne <measurement><amountUsed value="0.82 mg/l">0,82 mg/l</amountUsed> o <measuringTime value="03:10">03.10 h</measuringTime></measurement>.</offense>
"""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": sentence},
    ]
    tagged = LLM.chat(messages)


    return tagged


In [457]:
def add_structure_to_tags(id, sentence):
    system_prompt = f"""
You are an information extraction agent.

Your task is to insert only container tags into the input text in order to group already tagged attributes into logical event structures.

Return only the modified text.
Do not explain anything.

GENERAL RULES

Preserve the original text exactly.
Do not rewrite, translate, or change any words.

Existing tags must remain unchanged.
Do not modify, move, or delete any existing tags.

You may only add container tags.

Tags must not overlap or cross.

Nesting is allowed only for container tags.

Insert container tags so that related attributes are grouped into coherent events.

If container boundaries are unclear, do not add a container.

CONTAINER TAGS

You may insert only the following tags:

priorDecision
offense
measurement
vehicle
location
sanction
relatedOffense

Do not insert any other tags.

CONTAINER STRUCTURE

Use these containers when possible:

offense
Groups attributes describing a driving offense.

vehicle
Groups vehicleType and vehicleModel.

location
Groups placeOfOffense, placeDetails, geographicalDetails, and typeOfRoad.

measurement
Groups measuringTime, amountUsed, substanceUsed, measuringMethod, and measuringDevice belonging to one measurement event.

priorDecision
Groups information describing a previous administrative or court decision.

relatedOffense
Groups attributes describing the offense mentioned in a prior decision.

sanction
Groups sanction-related attributes.

GROUPING RULES

If vehicleType and vehicleModel appear together, wrap them inside a vehicle container.

If placeOfOffense, placeDetails, geographicalDetails, or typeOfRoad appear together, wrap them inside a location container.

If measuringTime, amountUsed, substanceUsed, measuringMethod, or measuringDevice appear together describing one measurement event, wrap them inside a measurement container.

If the text clearly describes one driving event, wrap the entire event in an offense container.

If multiple measurement events appear, create a separate measurement container for each measurement.

STRUCTURE PLANNING (DO NOT OUTPUT)

Before producing the output:

Identify the main offense event in the sentence.

Identify all measurement events.

Identify vehicle information.

Identify location information.

Insert container tags accordingly.

Ensure tags do not overlap or cross.

Do not output this reasoning.

FINAL VALIDATION

Before returning the result verify that:

all opening tags have closing tags

tags are properly nested

no tags overlap or cross

existing tags remain unchanged

Return only the tagged text.

Example

Input

dňa <dateOfOffense value="21.11.2022">21.11.2022</dateOfOffense> v čase <timeOfOffense value="00:15">00:15 h</timeOfOffense> v obci <placeOfOffense>Dunajská Lužná</placeOfOffense> na <placeDetails>Hlavnej ulici</placeDetails> viedol <vehicleType class="car">osobné motorové vozidlo</vehicleType> značky <vehicleModel>Škoda Octavia</vehicleModel>.

Output

<offense>dňa <dateOfOffense value="21.11.2022">21.11.2022</dateOfOffense> v čase <timeOfOffense value="00:15">00:15 h</timeOfOffense> v <location>obci <placeOfOffense>Dunajská Lužná</placeOfOffense> na <placeDetails>Hlavnej ulici</placeDetails></location> viedol <vehicle><vehicleType class="car">osobné motorové vozidlo</vehicleType> značky <vehicleModel>Škoda Octavia</vehicleModel></vehicle>.</offense>
"""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": sentence},
    ]
    tagged = LLM.chat(messages)


    return tagged


In [450]:
def extract_values_to_tags(id, tagged, classification_schema, formatting_rules):
    system_prompt = f"""
You are an information extraction and normalization agent.

You will receive text containing XML-like entity tags, for example:
<vehicleType>...</vehicleType>

Your task is to enrich existing opening tags by adding tag properties based on the rules below.

GENERAL RULES
- Preserve the original text exactly.
- Do not rewrite text outside tags.
- Do not change tag names.
- Do not remove tags.
- Do not add new tags.
- Do not change closing tags.
- Only add tag properties to opening tags.
- Return only the transformed text.
- Do not explain anything.

DECISION ORDER
For each tagged entity:
1. If the tag name is in the classification schema, add class="..."
2. Else if the tag name is in the formatting rules, add value="..."
3. Else if the content is anonymized and the tag has no classification rule, add value="anonymized"
4. Else add nothing

Do not add both class and value unless explicitly required. Normally a tag gets only one of them.

CLASSIFICATION SCHEMA

{format_value_schema(classification_schema)}

CLASSIFICATION INSTRUCTIONS
- Choose exactly one class from the allowed categories for that tag.
- Use tag content and local context if needed.
- If no category matches clearly, use class="other" only when that category exists.
- Otherwise add no class property.

FORMATTING RULES

{format_formatting_rules(formatting_rules)}

FORMATTING INSTRUCTIONS
- Add tag property value with the normalized value.
- Keep the original tagged text unchanged.

ANONYMIZATION RULE
- If the content is anonymized and the tag has no classification rule, add value="anonymized"
- Examples include initials, masked names, redacted strings, placeholders, or obvious anonymization markers

EXAMPLE

INPUT:
dňa <dateOfOffense>25. januára 2018</dateOfOffense> v čase okolo <timeOfOffense>02.50 h</timeOfOffense> v Stupave na Železničnej ulici, ako vodič viedol <vehicleType>osobné motorové vozidlo</vehicleType> značky <vehicleModel>O. T.</vehicleModel>, pričom u vodiča bola vykonaná <measuringMethod>dychová skúška</measuringMethod> prístrojom <measuringDevice>AlcoQuant 6020 plus</measuringDevice>, ktorou bola zistená hodnota <amountUsed>0,87 mg/l</amountUsed> o <measuringTime>02.55 h</measuringTime>.

OUTPUT:
dňa <dateOfOffense value="25.01.2018">25. januára 2018</dateOfOffense> v čase okolo <timeOfOffense value="02:50">02.50 h</timeOfOffense> v Stupave na Železničnej ulici, ako vodič viedol <vehicleType class="car">osobné motorové vozidlo</vehicleType> značky <vehicleModel value="anonymized">O. T.</vehicleModel>, pričom u vodiča bola vykonaná <measuringMethod class="breath">dychová skúška</measuringMethod> prístrojom <measuringDevice>AlcoQuant 6020 plus</measuringDevice>, ktorou bola zistená hodnota <amountUsed value="0.87 mg/l">0,87 mg/l</amountUsed> o <measuringTime value="02:55">02.55 h</measuringTime>.

"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": tagged},
    ]
    tagged = LLM.chat(messages)


    return tagged


In [451]:
def extract_values_to_json(id, tagged, classification_schema, formatting_rules):
    system_prompt = f"""
You are an information extraction and normalization agent.

You will receive text containing XML-like entity tags, for example:
<vehicleType>...</vehicleType>

Your task is to extract each tagged entity and return a JSON object describing it according to the rules below.

GENERAL RULES
- Preserve the original tag content exactly when reporting spans.
- Do not rewrite the text.
- Do not modify the tags.
- Only extract information from the tags.
- Return only a valid JSON object.
- Do not explain anything.

OUTPUT FORMAT

Return a JSON object where:
- each key is a tag name
- each value is a dictionary containing:
  - "span": the original tag content
  - optionally "class"
  - optionally "value"

Example:
{{
  "amountUsed": {{
    "span": "25 gramov",
    "value": "25 g"
  }}
}}

If multiple occurrences of the same tag appear in the input, the value must be a list of dictionaries:

Example:
{{
  "amountUsed": [
    {{"span": "0,87 mg/l", "value": "0.87 mg/l"}},
    {{"span": "0,82 mg/l", "value": "0.82 mg/l"}}
  ]
}}

DECISION ORDER
For each tagged entity:
1. If the tag name is in the classification schema, add "class"
2. Else if the tag name is in the formatting rules, add "value"
3. Else if the content is anonymized and the tag has no classification rule, add "value": "anonymized"
4. Else include only the "span"

Normally a tag gets only one of:
- "class"
- "value"

CLASSIFICATION SCHEMA

{format_value_schema(classification_schema)}
    
CLASSIFICATION INSTRUCTIONS
- Choose exactly one class from the allowed categories for that tag.
- Use tag content and local context if needed.
- If no category matches clearly, use "other" only when that category exists.
- Otherwise do not add "class".

FORMATTING RULES

{format_formatting_rules(formatting_rules)}

FORMATTING INSTRUCTIONS
- "span" must always contain the original tag content.
- "value" must contain the normalized value.

ANONYMIZATION RULE
- If the content is anonymized and the tag has no classification rule, add "value": "anonymized".
- Examples include initials, masked names, redacted strings, placeholders, or obvious anonymization markers.

EXAMPLE

INPUT:
dňa <dateOfOffense>25. januára 2018</dateOfOffense> v čase okolo <timeOfOffense>02.50 h</timeOfOffense> v Stupave na Železničnej ulici, ako vodič viedol <vehicleType>osobné motorové vozidlo</vehicleType> značky <vehicleModel>O. T.</vehicleModel>, pričom u vodiča bola vykonaná <measuringMethod>dychová skúška</measuringMethod> prístrojom <measuringDevice>AlcoQuant 6020 plus</measuringDevice>, ktorou bola zistená hodnota <amountUsed>0,87 mg/l</amountUsed> o <measuringTime>02.55 h</measuringTime>.

OUTPUT:
{{
  "dateOfOffense": {{
    "span": "25. januára 2018",
    "value": "25.01.2018"
  }},
  "timeOfOffense": {{
    "span": "02.50 h",
    "value": "02:50"
  }},
  "vehicleType": {{
    "span": "osobné motorové vozidlo",
    "class": "car"
  }},
  "vehicleModel": {{
    "span": "O. T.",
    "value": "anonymized"
  }},
  "measuringMethod": {{
    "span": "dychová skúška",
    "class": "breath"
  }},
  "amountUsed": {{
    "span": "0,87 mg/l",
    "value": "0.87 mg/l"
  }},
  "measuringTime": {{
    "span": "02.55 h",
    "value": "02:55"
  }}
}}
"""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": tagged},
    ]
    tagged = LLM.chat(messages)


    return tagged


In [452]:
def extract_values_complex(id, sentence, attribute_schema, classification_schema, formatting_rules):
    system_prompt = f"""
    Extract the following attributes into a JSON structure.
    Go through the given sentence and return JSON with the following structure:
    {{"attribute_name1":{{"span": "...", "value": "...", "class": "..."}}, ...}}

    {format_attribute_schema(attribute_schema)}
    
    Do not include redundant words (‘the day of’, ‘around’, ‘at the time’, etc.) unless they are part of the meaning.
    If a value is anonymized (e.g., XX.XX.XXXX, anonymized), still extract it. Anonymized data are extracted including the period.
    If attribute is of type "single", its value is a single contiguous text excerpt. If the relevant information is spread across the text, extract only the first part of the information.
    If attribute is of type "multiple", extract separate pieces of information separately in a list with preserved order (e.g. 0.5 mg/l of alcohol and 0.001 ng/l of THC ==> [0.5 mg/l ,  0.001 ng/l], [alcohol, THC])
    If an identical piece of information is repeated, do not extract it again.
    
    When relevant, add attribute class and/or value to the JSON. The category classes are given in the following schema. 
    If no categories are provided, follow the formatting guidelines. 

    CLASSIFICATION SCHEMA
    {format_value_schema(classification_schema)}

    FORMATTING RULES
    {formatting_rules}
    
    When a list of categories is defined for an attribute, pick the correct category and add it to the "class" key.
    When a formatting rule is defined, apply it and add it to the "value" key.
    If the tag content is anonymized and no class rule is defined, then value="anonymized"
    In no guidelines are provided, do not add add neither class nor value.
    
    Example annotations (use minimal span tagging with XML tags):

    on <dateOfOffence value="04.01.2022">04.01.2022</dateOfOffence> he was driving a motor vehicle
    Note: Do not include extra words such as "on".
    
    he was driving a passenger vehicle <vehicleModel>Škoda Octavia</vehicleModel>
    Note: Label only the vehicle model, not the full description.
    
    after previously consuming a large amount of <substanceUsed class="alcohol">alcohol</substanceUsed>
    Note: Label only the type of substance.
    
    on a <typeOfRoad class="class_I">I. class</typeOfRoad> road in the direction of Prešov
    Note: Include the road class designation.
    
    a driver who had a <licenseStatus class="banned">ban on driving motor vehicles</licenseStatus> imposed
    Note: Label the phrase that clearly indicates a driving prohibition.
    
    on <dateOfOffence value="anonymized">XX.XX.XXXX</dateOfOffence> he was driving a motor vehicle
    Note: Even if the value is anonymized (e.g., XX.XX.XXXX or anonymized), it must still be labeled with value=anonymized. Include punctuation as part of the labeled span.

    e.g. 
    INPUT:
    dňa 25. januara 2018 v čase okolo 02.50 h v Stupave na Železničnej ulicu, ako vodič viedol osobné motorové vozidlo mačky O. T., pričom u vodiča bola vykonaná dychová skúška prístrojom AlcoQuant 6020 plus, ktorou bola zistená hodnota 0,87 mg/l o 02.55 h a následne opakovaným meraním hodnota 0,82 mg/l o 03.10 h.
    OUTPUT:
    {{
          "dateOfOffense": {{
            "span": "25. januara 2018",
            "value": "25.01.2026",
          }},
          "timeOfOffense": {{
            "span": "02.50 h",
            "value": "02:50",
          }},
          "vehicleType": {{
            "span": "osobné motorové vozidlo",
            "class": "car"
          }},
          "vehicleModel": {{
            "span": "O. T.",
            "value": "O. T.",
          }},
          "measuringMethod": {{
            "span": "dychová skúška",
            "class": breath
          }},
          "measuringDevice": {{
            "span": "AlcoQuant 6020 plus",
            "value": "AlcoQuant 6020 plus",
          }},
          "amountUsed": [
            {{
              "span": "0,87 mg/l",
              "value": "0.87 mg/l",
            }},
            {{
              "span": "0,82 mg/l",
              "value": "0.82 mg/l",
            }}
          ],
          "measuringTime": [
            {{
              "span": "02.55",
              "value": "02:55",

            }},
            {{
              "span": "03.10",
              "value": "03:10",
            }}
          ]
    }}
"""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": sentence},
    ]
    tagged = LLM.chat(messages)


    return tagged


## Run extraction

In [935]:
LLM = get_llm("openai", model="gpt-5.1")  #get_llm("groq") #get_llm("openai", model="gpt-5.1") 

In [598]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def _task_tag_complex_with_structure(i):
    row = pod_vlivem.iloc[i]
    _id = int(row["judgement_slt_id"])
    sentence = row["judgement_factual_sentence"]
    res = tag_sentence_complex_with_structure(_id, sentence, attribute_schema, classification_schema, formatting_rules)
    return i, _id, res

results_tag_complex_with_structure = {}
errors_tag_complex_with_structure = {}

with ThreadPoolExecutor(max_workers=20) as ex:
    futures = {ex.submit(_task_tag_complex_with_structure, i): i for i in range(len(pod_vlivem))}
    for fut in as_completed(futures):
        i = futures[fut]
        try:
            _, _id, res = fut.result()
            results_tag_complex_with_structure[_id] = res
        except Exception as e:
            errors_tag_complex[i] = repr(e)

print(f"Done. Tagged: {len(results_tag_complex_with_structure)} | Errors: {len(errors_tag_complex_with_structure)}")

Done. Tagged: 201 | Errors: 0


In [874]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def _task_tag_complex(i):
    row = pod_vlivem.iloc[i]
    _id = int(row["judgement_slt_id"])
    sentence = row["judgement_factual_sentence"]
    res = tag_sentence_complex(_id, sentence, attribute_schema, classification_schema, formatting_rules)
    return i, _id, res

results_tag_complex = {}
errors_tag_complex = {}

with ThreadPoolExecutor(max_workers=20) as ex:
    futures = {ex.submit(_task_tag_complex, i): i for i in range(len(pod_vlivem))}
    for fut in as_completed(futures):
        i = futures[fut]
        try:
            _, _id, res = fut.result()
            results_tag_complex[_id] = res
        except Exception as e:
            errors_tag_complex[i] = repr(e)

print(f"Done. Tagged: {len(results_tag_complex)} | Errors: {len(errors_tag_complex)}")


You are an information extraction and normalization agent.

Your task is to insert XML-like tags into the input text and, when applicable, add tag properties class and/or value to the opening tag.

Use only the tag names defined below. Do not use any other tags. Return only the tagged text. Do not explain anything.

GENERAL RULES

Preserve the original text exactly.
Insert tags directly around the relevant span.
Do not rewrite, translate, or otherwise change the text content.
Tags must not overlap, cross, or nest.
Tag only the minimal span that expresses the information.
Do not include redundant words such as "on", "around", "at the time", etc., unless they are part of the meaning.
If several spans seem possible, choose the shortest span that still fully expresses the target attribute.
A single tagged span must represent one attribute only.
Tag only information explicitly stated in the text. Do not infer missing information.
If no listed attribute is present for a span, do not tag it.

In [722]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def _task_tag_complex(i):
    row = pod_vlivem.iloc[i]
    _id = int(row["judgement_slt_id"])
    sentence = row["judgement_factual_sentence"]
    res = tag_sentence_complex(_id, sentence, attribute_schema, classification_schema, formatting_rules)
    return i, _id, res

results_tag_complex = {}
errors_tag_complex = {}

for i in range(len(pod_vlivem)):
    try:
        _, _id, res = _task_tag_complex(i)
        print(f"{i}: success")
        results_tag_complex[_id] = res
    except Exception as ee:
        print(ee)
        print("error - waiting")
        time.sleep(30)
        _, _id, res = _task_tag_complex(i)
        try:
            results_tag_complex[_id] = res
        except Exception as e:
            print("error - moving on")
            errors_tag_complex[i] = repr(e)
    
print(f"Done. Tagged: {len(results_tag_complex)} | Errors: {len(errors_tag_complex)}")

0: success
1: success
2: success
3: success
4: success
5: success
6: success
7: success
8: success
9: success
10: success
11: success
12: success
13: success
14: success
15: success
16: success
17: success
18: success
19: success
20: success
21: success
22: success
23: success
24: success
25: success
26: success
27: success
28: success
29: success
30: success
31: success
32: success
33: success
34: success
35: success
36: success
37: success
38: success
39: success
40: success
41: success
42: success
43: success
44: success
45: success
46: success
47: success
48: success
49: success
50: success
51: success
52: success
53: success
54: success
55: success
56: success
57: success
58: success
59: success
60: success
61: success
62: success
63: success
64: success
65: success
66: success
67: success
68: success
69: success
70: success
71: success
72: success
73: success
74: success
75: success
76: success
77: success
78: success
79: success
80: success
81: success
82: success
83: success
84

In [914]:
import time


def _task_tag_in_two_steps(i):
    row = pod_vlivem.iloc[i]
    _id = row["judgement_slt_id"]
    sentence = row["judgement_factual_sentence"]

    tagged = tag_sentence(_id, sentence, attribute_schema)
    extracted = extract_values_to_tags(_id, tagged, classification_schema, formatting_rules)

    return i, _id, tagged, extracted

tagged_sentences = {}
results_tag_in_two_steps = {}
errors_tag_in_two_steps = {}

for i in range(len(pod_vlivem)):
    try:
        _, _id, tagged, res = _task_tag_in_two_steps(i)
        tagged_sentences[_id] = tagged
        print(f"{i}: success")
        results_tag_in_two_steps[_id] = res
    except Exception as ee:
        print(ee)
        print("error - waiting")
        time.sleep(30)
        _, _id, res = _task_tag_in_two_steps(i)
        try:
            results_tag_in_two_steps[_id] = res
        except Exception as e:
            print("error - moving on")
            errors_tag_in_two_steps[i] = repr(e)
    
print(f"Done. Tagged: {len(results_tag_in_two_steps)} | Errors: {len(errors_tag_in_two_steps)}")

0: success
1: success
2: success
3: success
4: success
5: success
6: success
7: success
8: success
9: success
10: success
11: success
12: success
13: success
14: success
15: success
16: success
17: success
18: success
19: success
20: success
21: success
22: success
23: success
24: success
25: success
26: success
27: success
28: success
29: success
30: success
31: success
32: success
33: success
34: success
35: success
36: success
37: success
38: success
39: success
40: success
41: success
42: success
43: success
44: success
45: success
46: success
47: success
48: success
49: success
50: success
51: success
52: success
53: success
54: success
55: success
56: success
57: success
58: success
59: success
60: success
61: success
62: success
63: success
64: success
65: success
66: success
67: success
68: success
69: success
70: success
71: success
72: success
73: success
74: success
75: success
76: success
77: success
78: success
79: success
80: success
81: success
82: success
83: success
84

In [ ]:
def _task_extract_complex(i):
    row = pod_vlivem.iloc[i]
    _id = row["judgement_slt_id"]
    sentence = row["judgement_factual_sentence"]
    res = extract_values_complex(_id, sentence, attribute_schema, classification_schema, formatting_rules)
    return i, _id, res

results_extract_complex = {}
errors_extract_complex = {}

for i in range(len(pod_vlivem)):
    try:
        _, _id, res = _task_extract_complex(i)
        print(f"{i}: success")
        results_extract_complex[_id] = res
    except Exception as ee:
        print(ee)
        print("error - waiting")
        time.sleep(30)
        _, _id, res = _task_extract_complex(i)
        try:
            errors_extract_complex[_id] = res
        except Exception as e:
            print("error - moving on")
            errors_tag_in_two_steps[i] = repr(e)
    
print(f"Done. Tagged: {len(results_extract_complex)} | Errors: {len(errors_extract_complex)}")

0: success
1: success
2: success
3: success
4: success
5: success
6: success
7: success
8: success
9: success
10: success
11: success
12: success
13: success
14: success
15: success
16: success
17: success
18: success
19: success
20: success
21: success
22: success
23: success
24: success
25: success
26: success
27: success
28: success
29: success
30: success
31: success
32: success
33: success
34: success
35: success
36: success
37: success
38: success
39: success
40: success
41: success
42: success
43: success
44: success
45: success
46: success
47: success
48: success
49: success
50: success
51: success
52: success
53: success
54: success
55: success
56: success
57: success
58: success
59: success
60: success
61: success
62: success
63: success
64: success
65: success
66: success
67: success
68: success
69: success
70: success
71: success
72: success
73: success
74: success
75: success
76: success
77: success
78: success
79: success
80: success
81: success
82: success
83: success
84

In [617]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def _task_tag_and_extract(i):
    row = pod_vlivem.iloc[i]
    _id = row["judgement_slt_id"]
    sentence = row["judgement_factual_sentence"]

    tagged = tag_sentence(_id, sentence, attribute_schema)
    extracted = extract_values_to_json(_id, tagged, classification_schema, formatting_rules)

    return i, _id, tagged, extracted

tagged_sentences = {}
results_tag_and_extract = {}
errors_tag_and_extract = {}

with ThreadPoolExecutor(max_workers=20) as ex:
    futures = {ex.submit(_task_tag_and_extract, i): i for i in range(len(pod_vlivem))}
    for fut in as_completed(futures):
        i = futures[fut]
        try:
            _, _id, tagged, extracted = fut.result()
            tagged_sentences[_id] = tagged
            results_tag_and_extract[_id] = extracted
        except Exception as e:
            errors_tag_and_extract[i] = repr(e)

print(f"Done. Tag+Extract: {len(results_tag_and_extract)} | Errors: {len(errors_tag_and_extract)}")

Done. Tag+Extract: 201 | Errors: 0


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def _task_tag_in_three_steps(i):
    row = pod_vlivem.iloc[i]
    _id = row["judgement_slt_id"]
    sentence = row["judgement_factual_sentence"]

    tagged = results_tag_in_two_steps[_id]
    extracted = add_structure_to_tags(_id, tagged)

    return i, _id, tagged, extracted

results_tag_in_three_steps = {}
errors_tag_in_three_steps = {}

with ThreadPoolExecutor(max_workers=20) as ex:
    futures = {ex.submit(_task_tag_in_three_steps, i): i for i in range(len(pod_vlivem))}
    for fut in as_completed(futures):
        i = futures[fut]
        try:
            _, _id, tagged, extracted = fut.result()
            results_tag_in_three_steps[_id] = extracted
        except Exception as e:
            errors_tag_in_three_steps[i] = repr(e)

print(f"Done. Tag in 3 steps: {len(results_tag_in_three_steps)} | Errors: {len(errors_tag_in_three_steps)}")

In [176]:
import pickle
with open("/Users/ivana/mff/gacr_pf/legal-attribute-extraction/results/results_tag_complex.pkl", "wb") as f:
    pickle.dump(results_tag_complex, f)
with open("/Users/ivana/mff/gacr_pf/legal-attribute-extraction/results/results_extract_complex.pkl", "wb") as f:
    pickle.dump(results_extract_complex, f)
with open("/Users/ivana/mff/gacr_pf/legal-attribute-extraction/results/results_tag_and_extract.pkl", "wb") as f:
    pickle.dump(results_tag_and_extract, f)
with open("/Users/ivana/mff/gacr_pf/legal-attribute-extraction/results/results_tag_in_two_steps.pkl", "wb") as f:
    pickle.dump(results_tag_in_three_steps, f)
with open("/Users/ivana/mff/gacr_pf/legal-attribute-extraction/results/results_tag_in_three_steps.pkl", "wb") as f:
    pickle.dump(results_tag_in_two_steps, f)
with open("/Users/ivana/mff/gacr_pf/legal-attribute-extraction/results/results_tag_complex_with_structure.pkl", "wb") as f:
    pickle.dump(results_tag_complex_with_structure, f)

## Transform to JSON

In [604]:
import re
import json
from collections import defaultdict
from html import unescape

TAG_RE = re.compile(
    r"<(?P<tag>[A-Za-z_][\w\-]*)"
    r"(?P<attrs>\s+[^>]*)?>"
    r"(?P<span>.*?)"
    r"</\1>",
    re.DOTALL
)

ATTR_RE = re.compile(r'(?P<key>[A-Za-z_][\w\-]*)\s*=\s*"(?P<val>[^"]*)"')

def tagged_text_to_json_dict(text: str) -> dict:
    acc = defaultdict(list)

    for m in TAG_RE.finditer(text):
        tag = m.group("tag")
        attrs_raw = m.group("attrs") or ""
        span = unescape(m.group("span")).strip()

        attrs = {a.group("key"): a.group("val") for a in ATTR_RE.finditer(attrs_raw)}

        item = {
            "span": span,
            "value": attrs.get("value", "n/a"),
            "class": attrs.get("class", "n/a"),
        }

        acc[tag].append(item)

    # collapse single-occurrence tags
    result = {}
    for tag, items in acc.items():
        result[tag] = items[0] if len(items) == 1 else items

    return result


In [729]:
results_tag_complex_json = {}
for id, sent in results_tag_complex.items():
    dicts = tagged_text_to_json_dict(sent)
    results_tag_complex_json[id] = dicts

In [620]:
results_tag_in_two_steps_json = {}
for id, sent in results_tag_in_two_steps.items():
    dicts = tagged_text_to_json_dict(sent)
    results_tag_in_two_steps_json[id] = dicts

In [621]:
import json
results_extract_complex_json = {id: json.loads(s) for id, s in results_extract_complex.items()}
results_tag_and_extract_json = {id: json.loads(s) for id, s in results_tag_and_extract.items()}

## View values per tag

In [622]:
import numpy as np
def transform_per_tag(attributes_per_sentence):
    attributes_per_tag = {}
    i = 0
    for id, tags in attributes_per_sentence.items():
        for tag, values in tags.items():
            
            
            if not isinstance(values, list):
                values = [values]
            for val in values:
                attributes_per_tag[i] = {}
                attributes_per_tag[i]["name"] = tag
                if val == "n/a": # Fix extraction errors
                    val = {}
                attributes_per_tag[i]["span"] = val.get("span", "n/a")
                attributes_per_tag[i]["value"] = val.get("value", "n/a")
                attributes_per_tag[i]["class"] = val.get("class", "n/a")
                attributes_per_tag[i]["sentence_id"] = id
                attributes_per_tag[i]["sentence_raw"] = pod_vlivem[pod_vlivem["judgement_slt_id"] == np.int64(id)]["judgement_factual_sentence"].iloc[0]
                attributes_per_tag[i]["sentence_tagged"] = attributes_per_sentence[id]
    
                i += 1
    return attributes_per_tag



In [628]:
long_results_tag_complex = transform_per_tag(results_tag_complex_json)
long_results_extract_complex = transform_per_tag(results_extract_complex_json)
long_results_tag_and_extract = transform_per_tag(results_tag_and_extract_json)
long_results_tag_in_two_steps = transform_per_tag(results_tag_in_two_steps_json)

## Transform to flat tables for comparison

In [624]:
def export(attr_dict, name):
    df = pd.DataFrame.from_dict(attr_dict, orient="index").reset_index()
    df.to_csv(f"~/mff/gacr_pf/legal-attribute-extraction/results/{name}.csv", index=False)
    return df

In [630]:
import pandas as pd

FIELDS = ["span", "value", "class"]

def max_counts(docs):
    counts = {}
    for doc in docs:
        for tag, content in doc.items():
            c = len(content) if isinstance(content, list) else 1
            counts[tag] = max(counts.get(tag, 0), c)
    return counts

def flatten_doc_wide(doc, max_per_tag, fill="n/a"):
    row = {}
    for tag, max_c in max_per_tag.items():
        content = doc.get(tag, None)

        items = []
        if content is None:
            items = []
        elif isinstance(content, list):
            items = content
        else:
            items = [content]

        # fill up to max_c
        for i in range(1, max_c + 1):
            item = items[i - 1] if i - 1 < len(items) else None
            if item == "n/a": # Fix extraction errors
                item = {}
            for f in FIELDS:
                col = f"{tag}_{i}_{f}" if max_c > 1 else f"{tag}_{f}"
                row[col] = item.get(f, fill) if item else fill

    return row


max_per_tag1 = max_counts(results_tag_complex_json.values())
max_per_tag2 = max_counts(results_extract_complex_json.values())
max_per_tag3 = max_counts(results_tag_and_extract_json.values())
#max_per_tag = {tag: max(max_per_tag1[tag], max_per_tag2[tag], max_per_tag3[tag]) for tag in max_per_tag1}
max_per_tag = max_per_tag2

flat_results_tag_complex = {id: flatten_doc_wide(doc, max_per_tag, fill="n/a") for id, doc in results_tag_complex_json.items()}
flat_results_extract_complex = {id: flatten_doc_wide(doc, max_per_tag, fill="n/a") for id, doc in results_extract_complex_json.items()}
flat_results_tag_and_extract = {id: flatten_doc_wide(doc, max_per_tag, fill="n/a") for id, doc in results_tag_and_extract_json.items()}
flat_results_tag_in_two_steps = {id: flatten_doc_wide(doc, max_per_tag, fill="n/a") for id, doc in results_tag_in_two_steps_json.items()}

In [626]:
import pandas as pd
import re

def aggregate_tag_columns(
    df: pd.DataFrame,
    prefix: str,
    field: str,
    output_col: str | None = None,
    sep: str = ";",
    sort: bool = True,
) -> pd.DataFrame:
    """
    Aggregate columns like:
        {prefix}_1_{field}, {prefix}_2_{field}, ...

    Example:
        prefix='substanceUsed', field='value'
        -> substanceUsed_1_value, substanceUsed_2_value, ...

    Creates a new column with deduplicated joined values.
    """
    cols = df.filter(regex=rf"^{re.escape(prefix)}_\d+_{re.escape(field)}$").columns

    if len(cols) == 0:
        if output_col is None:
            output_col = f"{prefix}_{field}_agg"
        df[output_col] = ""
        return df

    if output_col is None:
        output_col = f"{prefix}_{field}"

    def compile_row(row):
        values = {
            str(v).strip()
            for v in row[cols]
            if not isinstance(v,list) and (pd.notna(v) and str(v).strip() != "" and str(v).strip() != "n/a" and str(v).strip() != "None")
        }
        if sort:
            return sep.join(sorted(values, key=str.lower))
        return sep.join(values)

    df[output_col] = df.apply(compile_row, axis=1)
    return df

In [159]:
df_flat_results_tag_complex = export(flat_results_tag_complex, "eval_flat_results_tag_complex_gpt-5-mini")
df_flat_results_extract_complex = export(flat_results_extract_complex, "eval_flat_results_extract_complex_gpt-5-mini")
df_flat_results_tag_and_extract = export(flat_results_tag_and_extract, "eval_flat_results_tag_and_extract_gpt-5-mini")
df_flat_results_tag_in_two_steps = export(flat_results_tag_in_two_steps, "eval_flat_results_tag_in_two_steps_gpt-5-mini")

df_long_results_tag_complex = export(long_results_tag_complex, "eval_long_results_tag_complex_gpt-5-mini")
df_long_results_extract_complex = export(long_results_extract_complex, "eval_long_results_extract_complex_gpt-5-mini")
df_long_results_tag_and_extract = export(long_results_tag_and_extract, "eval_long_results_tag_and_extract_gpt-5-mini")
df_long_results_tag_in_two_steps = export(long_results_tag_in_two_steps, "eval_long_results_tag_in_two_steps_gpt-5-mini")
